In [1]:
import os
import uuid
from datetime import datetime
import json

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
import pyspark.sql.functions as F

## Reading Parquet files from the data directory

In [2]:
INBOX_DIR = "/home/jovyan/data/inbox"

spark = SparkSession.builder.appName("Project1").getOrCreate()


def list_parquet_files(inbox_dir: str):
    if not os.path.isdir(inbox_dir):
        return []

    return sorted(
        os.path.join(inbox_dir, name)
        for name in os.listdir(inbox_dir)
        if name.endswith(".parquet")
    )


parquet_files = list_parquet_files(INBOX_DIR)
print(f"Parquet files to read: {len(parquet_files)}")

Parquet files to read: 2


In [3]:
parquet_files

['/home/jovyan/data/inbox/yellow_tripdata_2025-01.parquet',
 '/home/jovyan/data/inbox/yellow_tripdata_2025-02.parquet']

## Verifying whether files have been processed

In [4]:
MANIFEST_PATH = "/home/jovyan/state/manifest.json"

if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, "r") as f:
        manifest = json.load(f)
    
    processed_paths = {
        file["path"]
        for batch in manifest.get("batches", [])
        for file in batch.get("processed_files", [])
    }
    parquet_files = [p for p in parquet_files if p not in processed_paths]

In [5]:
parquet_files

[]

# Mapping

In [6]:
if parquet_files:
    df = spark.read.option("mergeSchema", "true").parquet(*parquet_files).withColumn(
        "source_file", F.input_file_name()
    )
else:
    df = None
    print("No parquet files found; skipping.")

No parquet files found; skipping.


In [7]:
if df != None:
    df.show(5, truncate=False)


In [8]:
if df != None:
    df.describe()

## Zone lookup

In [9]:
lookup_files = spark.read.option("mergeSchema", "true").parquet(
    f"{INBOX_DIR}/lookup/*.parquet")

zone_lookup = (
    lookup_files
    .filter(F.col("LocationID").isNotNull() & F.col("Zone").isNotNull())
    .select(
        F.col("LocationID").cast("int").alias("LocationID"),
        F.col("Zone").alias("zone_name")
    )
    .dropDuplicates(["LocationID"])
)

In [10]:
zone_lookup.show(5, truncate=False)

+----------+-----------------------+
|LocationID|zone_name              |
+----------+-----------------------+
|1         |Newark Airport         |
|2         |Jamaica Bay            |
|3         |Allerton/Pelham Gardens|
|4         |Alphabet City          |
|5         |Arden Heights          |
+----------+-----------------------+
only showing top 5 rows


## Mapping

In [11]:
if df != None:
    df.filter(
        F.col("fare_amount") < 0
    ).show(10, truncate=False)

In [12]:
if df != None:
    trips = df.filter(
        F.col("tpep_pickup_datetime").isNotNull()
        & F.col("tpep_dropoff_datetime").isNotNull()
        & F.col("PULocationID").isNotNull()
        & F.col("DOLocationID").isNotNull()
    )

In [13]:
if df != None:
    trips = df.filter(
        (F.col("tpep_pickup_datetime").isNotNull())
        & (F.col("tpep_dropoff_datetime").isNotNull())
        & (F.col("PULocationID").isNotNull())
        & (F.col("DOLocationID").isNotNull())
        & (F.col("passenger_count") >= 0)
        & (F.col("trip_distance") >= 0)
        & (F.col("fare_amount") >= 0)
        & (F.col("tip_amount") >= 0)
    )

# Join with trips data
    mapped_df = (
        trips.alias("t")
        .join(zone_lookup.alias("pu"), F.col("t.PULocationID") == F.col("pu.LocationID"), "left")
        .join(zone_lookup.alias("do"), F.col("t.DOLocationID") == F.col("do.LocationID"), "left")
        .select(
            F.col("t.tpep_pickup_datetime").alias("pickup_timestamp"),
            F.col("t.tpep_dropoff_datetime").alias("dropoff_timestamp"),
            F.col("t.PULocationID").alias("pickup_location_id"),
            F.col("t.DOLocationID").alias("dropoff_location_id"),
            F.col("pu.zone_name").alias("pickup_zone_name"),
            F.col("do.zone_name").alias("dropoff_zone_name"),
            F.col("t.passenger_count"),
            F.col("t.trip_distance"),
            F.round(
                (
                    (
                        F.unix_timestamp(
                            F.col("t.tpep_dropoff_datetime").cast("timestamp"))
                        - F.unix_timestamp(F.col("t.tpep_pickup_datetime").cast("timestamp"))
                    ) / 60.0
                ),
                2,
            ).alias("trip_duration_minutes"),
            F.to_date(F.col("t.tpep_pickup_datetime")).alias("pickup_date"),
            F.col("t.source_file"),
            F.current_timestamp().alias("ingested_at"),
        )
    )

    mapped_df.show(10, truncate=False)

## Examples of bad rows



In [14]:
if df != None:
    bad_trips = df.filter(
        (F.col("tpep_pickup_datetime").isNull())
        | (F.col("tpep_dropoff_datetime").isNull())
        | (F.col("PULocationID").isNull())
        | (F.col("DOLocationID").isNull())
        | (F.col("passenger_count").isNull() | (F.col("passenger_count") < 0))
        | (F.col("trip_distance").isNull() | (F.col("trip_distance") < 0))
        | (F.col("fare_amount").isNull() | (F.col("fare_amount") < 0))
        | (F.col("tip_amount").isNull() | (F.col("tip_amount") < 0))
    )
    bad_trips.show(10, truncate=False)

## Writing the Output


In [15]:
import glob
import shutil
if df != None:
    TEMP_DIR = "/home/jovyan/data/outbox/_temp_enriched"
    OUTBOX_PATH = "/home/jovyan/data/outbox/trips_enriched.parquet"

    if os.path.exists(OUTBOX_PATH):
        existing = spark.read.parquet(OUTBOX_PATH)
        combined = existing.unionByName(mapped_df, allowMissingColumns=True)
    else:
        combined = mapped_df

    # Write single file
    combined.coalesce(1).write.mode("overwrite").parquet(TEMP_DIR)
    parquet_file = glob.glob(f"{TEMP_DIR}/*.parquet")[0]
    shutil.move(parquet_file, OUTBOX_PATH)
    shutil.rmtree(TEMP_DIR)

    os.path.getsize(OUTBOX_PATH)

In [16]:
MANIFEST_PATH = "/home/jovyan/state/manifest.json"

if df != None:
    current_batch = {
        "batch_id": str(uuid.uuid4()),
        "created_at": datetime.now().isoformat(),
        "processed_files": [
            {
                "path": p,
                "size": os.path.getsize(p),
                "modified": datetime.fromtimestamp(os.path.getmtime(p)).isoformat(),
            }
            for p in parquet_files
        ],
        "file_size": sum(os.path.getsize(p) for p in parquet_files),
        "row_count": df.count(),
    }

    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r") as f:
            manifest = json.load(f)
        manifest.setdefault("batches", []).append(current_batch)
    else:
        manifest = {"batches": [current_batch]}

    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2)

    manifest

## Compute the top 5 pickup zones by total trip count

## Read the file

In [17]:
OUTBOX_DIR = "/home/jovyan/data/outbox"

spark = SparkSession.builder.appName("Project1").getOrCreate()

def list_parquet_files(outbox_dir: str):
    if not os.path.isdir(outbox_dir):
        return []

    return sorted(
        os.path.join(outbox_dir, name)
        for name in os.listdir(outbox_dir)
        if name.endswith("enriched.parquet")
    )


parquet_files = list_parquet_files(OUTBOX_DIR)
print(f"Parquet files to read: {len(parquet_files)}")
print(parquet_files)

if parquet_files:
    df = spark.read.parquet(parquet_files[0])
    df.describe()
    df.show(5)
else:
    df = None

Parquet files to read: 1
['/home/jovyan/data/outbox/trips_enriched.parquet']
+-------------------+-------------------+------------------+-------------------+--------------------+--------------------+---------------+-------------+---------------------+-----------+--------------------+--------------------+
|   pickup_timestamp|  dropoff_timestamp|pickup_location_id|dropoff_location_id|    pickup_zone_name|   dropoff_zone_name|passenger_count|trip_distance|trip_duration_minutes|pickup_date|         source_file|         ingested_at|
+-------------------+-------------------+------------------+-------------------+--------------------+--------------------+---------------+-------------+---------------------+-----------+--------------------+--------------------+
|2025-01-01 00:18:38|2025-01-01 00:26:59|               229|                237|Sutton Place/Turt...|Upper East Side S...|              1|          1.6|                 8.35| 2025-01-01|file:///home/jovy...|2026-03-07 11:03:...|
|2025-0

## Perform operations on the data

In [18]:
zones = spark.read.parquet("/home/jovyan/data/inbox/lookup/taxi_zone_lookup.parquet")
if df != None:
    top_zones = (
        df.groupBy("pickup_location_id", "pickup_zone_name")
        .agg(F.count("*").alias("trip_count"))
        .orderBy(F.col("trip_count").desc())
        .limit(5)
        .join(zones.select("LocationID", "Borough"), 
            F.col("pickup_location_id") == F.col("LocationID"), "left")
        .withColumnRenamed("pickup_zone_name", "zone")
        .withColumnRenamed("Borough", "borough")
        .select("zone", "borough", "trip_count")
    )
    top_zones.show()

+--------------------+---------+----------+
|                zone|  borough|trip_count|
+--------------------+---------+----------+
|Upper East Side S...|Manhattan|    291126|
|      Midtown Center|Manhattan|    287436|
|Upper East Side N...|Manhattan|    267806|
|         JFK Airport|   Queens|    253750|
|Penn Station/Madi...|Manhattan|    212020|
+--------------------+---------+----------+



## Write the file

In [19]:
import glob
import shutil

TEMP_DIR = "/home/jovyan/data/outbox/_temp_zones"
OUTBOX_PATH = "/home/jovyan/data/outbox/top_zones.parquet"

if top_zones:
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)


    top_zones.coalesce(1).write.mode("overwrite").parquet(TEMP_DIR)
    parquet_file = glob.glob(f"{TEMP_DIR}/*.parquet")[0]
    shutil.move(parquet_file, OUTBOX_PATH)
    shutil.rmtree(TEMP_DIR)

    print(f"Written to {OUTBOX_PATH}")
    spark.read.parquet(OUTBOX_PATH).show(truncate=False)

Written to /home/jovyan/data/outbox/top_zones.parquet
+----------------------------+---------+----------+
|zone                        |borough  |trip_count|
+----------------------------+---------+----------+
|Upper East Side South       |Manhattan|291126    |
|Midtown Center              |Manhattan|287436    |
|Upper East Side North       |Manhattan|267806    |
|JFK Airport                 |Queens   |253750    |
|Penn Station/Madison Sq West|Manhattan|212020    |
+----------------------------+---------+----------+

